# Brecha Digital Territorial en Cusco

Cruce de luces nocturnas (NASA VNL 2025) con cobertura móvil OSIPTEL 2019 para identificar zonas con brecha digital activa en la región Cusco.

Pipeline: carga raster → reproyección a EPSG:4326 → normalización p2-p98 → índices (IBD, EDT) → mapas temáticos → clasificación 2x2 → stats.

## Step 0 — Setup

Imports y versiones de las librerías para reproducibilidad.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.transform import array_bounds
import scipy
from scipy import ndimage, stats

print('rasterio  ', rasterio.__version__)
print('numpy     ', np.__version__)
print('matplotlib', plt.matplotlib.__version__)
print('scipy     ', scipy.__version__)
print('seaborn   ', sns.__version__)
print('pandas    ', pd.__version__)

sns.set_context('notebook')

# Rutas relativas. Soporta ejecución desde notebooks/ y desde la raiz del repo.
for candidate in [Path('../data'), Path('data')]:
    if candidate.exists():
        DATA = candidate.resolve()
        break
OUT = (DATA.parent / 'output')
OUT.mkdir(exist_ok=True)
print('DATA dir  ', DATA)
print('OUTPUT dir', OUT)

## Step 1 — Loading e inspección

Para cada raster imprimo CRS, shape, bandas, NoData, dtype, bbox, resolución y rango de valores.

In [ ]:
def km_per_pixel(transform, height, width, crs):
    """Aprox de tamaño de pixel en km, dependiendo del CRS."""
    dx, dy = abs(transform.a), abs(transform.e)
    if crs and crs.is_geographic:
        bounds = array_bounds(height, width, transform)
        lat_mid = 0.5 * (bounds[1] + bounds[3])
        kx = dx * 111.32 * np.cos(np.deg2rad(lat_mid))
        ky = dy * 110.57
        return dx, dy, kx, ky, 'deg'
    return dx, dy, dx / 1000.0, dy / 1000.0, 'm'


def inspect_raster(path, label):
    with rasterio.open(path) as ds:
        arr = ds.read(1, masked=True)
        dx, dy, kx, ky, unit = km_per_pixel(ds.transform, ds.height, ds.width, ds.crs)
        valid = int(arr.count())
        total = int(arr.size)
        amin = float(arr.min()) if valid else float('nan')
        amax = float(arr.max()) if valid else float('nan')
        print(f'=== {label} ===')
        print(f'  path     : {path.name}')
        print(f'  CRS      : {ds.crs}')
        print(f'  shape    : {ds.height} x {ds.width} (alto x ancho)')
        print(f'  bandas   : {ds.count}')
        print(f'  NoData   : {ds.nodata}')
        print(f'  dtype    : {ds.dtypes[0]}')
        print(f'  bbox     : {ds.bounds}')
        print(f'  pixel    : {dx:.6f} x {dy:.6f} {unit}  (~{kx*1000:.0f} x {ky*1000:.0f} m)')
        print(f'  validos  : {valid:,} / {total:,} pixeles')
        print(f'  rango    : [{amin:.6g}, {amax:.6g}]')
        print()
        return arr


VNL_PATH = DATA / 'VNL_cusco_2025.tif'
KERN_PATH = DATA / 'kernel_cobmovil2019_50m.tif'

vnl_arr = inspect_raster(VNL_PATH, 'VNL_cusco_2025 (NASA Black Marble)')
kern_arr = inspect_raster(KERN_PATH, 'kernel_cobmovil2019_50m (OSIPTEL)')

## Step 2 — Reproyección y alineación

El kernel OSIPTEL viene en EPSG:32719 (UTM 19S, metros). Lo reproyecto a EPSG:4326 con resampling bilineal y lo paso a la grilla exacta del VNL (mismo transform, mismo shape).

In [ ]:
with rasterio.open(VNL_PATH) as vnl_ds:
    vnl_transform = vnl_ds.transform
    vnl_crs = vnl_ds.crs
    vnl_shape = (vnl_ds.height, vnl_ds.width)
    vnl_bounds = vnl_ds.bounds
    vnl_data = vnl_ds.read(1).astype('float32')
    vnl_nodata = vnl_ds.nodata
    vnl_profile = vnl_ds.profile

with rasterio.open(KERN_PATH) as kern_ds:
    kern_src = kern_ds.read(1).astype('float32')
    kern_transform_src = kern_ds.transform
    kern_crs_src = kern_ds.crs
    kern_nodata_src = kern_ds.nodata

# Reproyecto el kernel directo a la grilla del VNL
conn = np.full(vnl_shape, np.nan, dtype='float32')
reproject(
    source=kern_src,
    destination=conn,
    src_transform=kern_transform_src,
    src_crs=kern_crs_src,
    dst_transform=vnl_transform,
    dst_crs=vnl_crs,
    resampling=Resampling.bilinear,
    src_nodata=kern_nodata_src,
    dst_nodata=np.nan,
)

assert conn.shape == vnl_data.shape, 'shapes no coinciden'
print(f'VNL shape           : {vnl_data.shape}')
print(f'Conn reproyectada   : {conn.shape}')
print(f'mismo grid?         : {conn.shape == vnl_data.shape}')
print(f'CRS final           : {vnl_crs}')
print(f'extent (W,S,E,N)    : {vnl_bounds}')
valid_conn = int(np.sum(~np.isnan(conn)))
print(f'pixeles validos conn: {valid_conn:,} / {conn.size:,}')

## Step 3 — Normalización robusta (p2 a p98)

Negativos y NaN/NoData los reemplazo por 0 antes del clip. Uso percentil [2, 98] para ser robusto contra outliers (un pico de VNL muy alto en el centro de la ciudad no debe saturar todo el rango). El resultado queda en [0, 1] en ambas capas.

In [ ]:
def normalize_p298(arr):
    a = np.where(np.isnan(arr) | (arr < 0), 0.0, arr).astype('float32')
    p2, p98 = np.percentile(a, [2, 98])
    out = np.clip((a - p2) / (p98 - p2 + 1e-12), 0.0, 1.0).astype('float32')
    return out, float(p2), float(p98)


vnl_norm, vnl_p2, vnl_p98 = normalize_p298(vnl_data)
conn_norm, conn_p2, conn_p98 = normalize_p298(conn)

for label, x, p2, p98 in [
    ('VNL_norm', vnl_norm, vnl_p2, vnl_p98),
    ('Conn_norm', conn_norm, conn_p2, conn_p98),
]:
    print(f'{label}:  p2={p2:.6g}, p98={p98:.6g}')
    print(f'   min={x.min():.4f}  max={x.max():.4f}  mean={x.mean():.4f}  std={x.std():.4f}')
    print()